# 11 · Stream Processing — When the Input Never Ends

**Part 1 · Data-Intensive Applications** · *Prerequisites: 05, 10* · *Book mapping: DDIA ch. 11*

Batch (nb 10) assumed a bounded input you can see all of. Reality emits **events** — clicks,
payments, sensor readings — continuously, forever. Stream processing is batch's unbounded
sibling: same derived-data factory, but the input is an endless, ordered, replayable **log**
(nb 03's append-only structure, promoted from storage-engine internals to *the* integration
fabric between systems).

**Learning objectives.**

1. Build a partitioned, log-based **message broker** (the Kafka model): offsets, consumer
   groups, load balancing, fan-out, replay.
2. Demonstrate all three delivery semantics — at-most-once, at-least-once, effectively-once —
   by crashing consumers at exactly the wrong moments.
3. Handle **time**: event time vs processing time, out-of-order arrival, windows, and
   watermarks — with the lateness trade-off measured.
4. Build a stream-table join from a **changelog**, and see event sourcing / CDC as the
   pattern underneath (setting up nb 12).

In [ ]:
import hashlib
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

from sysdes import viz

viz.use_style()
rng = np.random.default_rng(seed=11)

def stable_hash(s: str) -> int:
    return int.from_bytes(hashlib.blake2b(str(s).encode(), digest_size=8).digest())

## 1. The log-based broker

Two messaging philosophies exist. Classic **message queues** (AMQP/JMS) hand each message to
one consumer and *delete it once acknowledged* — good for task queues, but the data is gone.
**Log-based brokers** (Kafka, Kinesis) instead *append* every message to a partitioned,
durable log; consumers just remember **how far they've read** (an **offset**, one integer
per partition). Messages aren't deleted on read — so many independent groups can consume the
same stream, slow consumers hurt nobody, and *history is replayable*.

In [ ]:
fig, ax = viz.system_diagram(
    nodes=[
        {"id": "p1", "x": 0, "y": 1.7, "label": "producer A", "kind": "client"},
        {"id": "p2", "x": 0, "y": 0.3, "label": "producer B", "kind": "client"},
        {"id": "t0", "x": 2.7, "y": 2.0, "label": "partition 0 | 0 1 2 3 ...", "kind": "queue"},
        {"id": "t1", "x": 2.7, "y": 1.0, "label": "partition 1 | 0 1 2 ...", "kind": "queue"},
        {"id": "t2", "x": 2.7, "y": 0.0, "label": "partition 2 | 0 1 ...", "kind": "queue"},
        {"id": "g1", "x": 5.7, "y": 1.6, "label": "group 'billing'\n(2 consumers,\nsplit partitions)"},
        {"id": "g2", "x": 5.7, "y": 0.1, "label": "group 'analytics'\n(reads everything\nindependently)"},
    ],
    edges=[
        {"src": "p1", "dst": "t0", "label": "hash(key)"},
        {"src": "p1", "dst": "t1"},
        {"src": "p2", "dst": "t1"},
        {"src": "p2", "dst": "t2"},
        {"src": "t0", "dst": "g1", "label": "offset 4"},
        {"src": "t1", "dst": "g1"},
        {"src": "t1", "dst": "g2", "label": "offset 1", "style": "dashed"},
        {"src": "t2", "dst": "g1"},
        {"src": "t2", "dst": "g2", "style": "dashed"},
    ],
    title="A topic = partitioned append-only logs; a consumer = a set of offsets",
)
plt.show()

In [ ]:
class LogBroker:
    """A topic: N append-only partitions + committed offsets per consumer group."""

    def __init__(self, n_partitions: int = 3):
        self.partitions: list[list[dict]] = [[] for _ in range(n_partitions)]
        self.committed: dict[tuple[str, int], int] = {}   # (group, partition) -> offset

    def produce(self, key: str, value: dict) -> tuple[int, int]:
        """Same key -> same partition, ALWAYS: per-key order is preserved
        because one partition is one strictly-ordered log (nb 06 + nb 03)."""
        p = stable_hash(key) % len(self.partitions)
        self.partitions[p].append({"key": key, "value": value,
                                   "offset": len(self.partitions[p])})
        return p, len(self.partitions[p]) - 1

    def fetch(self, partition: int, offset: int) -> list[dict]:
        return self.partitions[partition][offset:]

    def commit(self, group: str, partition: int, offset: int) -> None:
        self.committed[(group, partition)] = offset

    def committed_offset(self, group: str, partition: int) -> int:
        return self.committed.get((group, partition), 0)

    def assign(self, members: list[str]) -> dict[str, list[int]]:
        """Round-robin partition assignment: each partition to EXACTLY ONE
        member of a group - that's how a group load-balances a topic."""
        assignment = {m: [] for m in members}
        for p in range(len(self.partitions)):
            assignment[members[p % len(members)]].append(p)
        return assignment


class Consumer:
    """One member of a consumer group. Reading position starts at the group's
    committed offsets - which is all a 'restart after crash' means."""

    def __init__(self, broker: LogBroker, group: str, partitions: list[int]):
        self.broker, self.group, self.partitions = broker, group, partitions
        self.position = {p: broker.committed_offset(group, p) for p in partitions}

    def poll(self) -> list[dict]:
        batch = []
        for p in self.partitions:
            records = self.broker.fetch(p, self.position[p])
            for r in records:
                batch.append({**r, "partition": p})
            self.position[p] += len(records)
        return batch

    def commit(self) -> None:
        for p, off in self.position.items():
            self.broker.commit(self.group, p, off)

broker = LogBroker(n_partitions=3)
for i in range(12):
    user = f"user{i % 4}"
    broker.produce(user, {"seq": i // 4, "user": user})   # per-user seq 0,1,2

# Group 'billing': two consumers SPLIT the partitions (load balancing)...
assignment = broker.assign(["c1", "c2"])
c1 = Consumer(broker, "billing", assignment["c1"])
c2 = Consumer(broker, "billing", assignment["c2"])
got1, got2 = c1.poll(), c2.poll()

# ...each message went to exactly one member, and together they saw everything:
ids1 = {(m["partition"], m["offset"]) for m in got1}
ids2 = {(m["partition"], m["offset"]) for m in got2}
assert ids1.isdisjoint(ids2) and len(ids1 | ids2) == 12, \
    "a group divides the stream: no overlap, no gaps"

# Group 'analytics' reads the SAME 12 messages, independently (fan-out):
c3 = Consumer(broker, "analytics", [0, 1, 2])
assert len(c3.poll()) == 12, "logs are not consumed away - every group gets a full copy"

# Per-key ordering: within any consumer, each user's seq numbers arrive in order.
for batch in (got1, got2):
    per_user = defaultdict(list)
    for m in batch:
        per_user[m["value"]["user"]].append(m["value"]["seq"])
    assert all(seqs == sorted(seqs) for seqs in per_user.values()), \
        "same key -> same partition -> that key's history is strictly ordered"
print("load balancing, fan-out, and per-key ordering all verified")

Two facts to internalize from that cell: **ordering exists only per partition** (global
order across partitions is undefined — choose your partition key to be the identity whose
history must stay ordered), and **a "restart" is nothing special** — a consumer is just its
group's committed offsets, so replay = reset the offsets:

In [ ]:
for p in range(3):
    broker.commit("analytics", p, 0)          # rewind the whole group to the start
assert len(Consumer(broker, "analytics", [0, 1, 2]).poll()) == 12, \
    "replay: reprocess history at will (new code, bug fix, new derived view - nb 12)"
print("rewound and replayed - the log remembers")

## 2. Delivery semantics — crash at the worst moment

A consumer does two things per message: **process** (side effect: send the email, update
the balance) and **commit** (record the offset). A crash between them is guaranteed
eventually, and the *order you chose* decides what happens:

In [ ]:
def make_email_topic():
    b = LogBroker(1)
    for i in range(5):
        b.produce("mail", {"email_id": f"e{i}", "to": f"user{i}"})
    return b

# --- AT-LEAST-ONCE: process, THEN commit. Crash before commit -> redelivery.
b = make_email_topic()
sent: list[str] = []
c = Consumer(b, "mailer", [0])
for msg in c.poll():
    sent.append(msg["value"]["email_id"])      # processed all 5...
# ...CRASH before c.commit(). The replacement consumer starts at offset 0:
c_new = Consumer(b, "mailer", [0])
for msg in c_new.poll():
    sent.append(msg["value"]["email_id"])
c_new.commit()
print(f"at-least-once: {len(sent)} emails sent for {len(set(sent))} messages")
assert len(sent) == 10 and len(set(sent)) == 5, \
    "every user got their email TWICE - duplicates are the price of never losing"

# --- AT-MOST-ONCE: commit FIRST, then process. Crash mid-processing -> loss.
b = make_email_topic()
sent = []
c = Consumer(b, "mailer2", [0])
batch = c.poll()
c.commit()                                     # offsets say 'done' already...
for msg in batch[:2]:
    sent.append(msg["value"]["email_id"])      # ...CRASH after only 2 of 5
c_new = Consumer(b, "mailer2", [0])
assert c_new.poll() == [], "nothing left to read - the offsets moved on"
print(f"at-most-once: {len(sent)} of 5 emails sent; 3 are gone forever")
assert len(sent) == 2

Neither is acceptable for money or email. The industry answer is **at-least-once delivery +
idempotent processing** — make the duplicate *harmless* rather than impossible. Every
message carries a stable id; the consumer records ids it has fully processed and skips
repeats. Result: **effectively once** (often marketed as "exactly-once semantics"):

In [ ]:
b = make_email_topic()
sent, processed_ids = [], set()

def handle(msg):
    """Idempotent handler: the side effect is guarded by the message's identity.
    (In production, `processed_ids` lives in the SAME transactional store as the
    side effect, so 'sent' and 'marked processed' commit atomically - nb 07.)"""
    eid = msg["value"]["email_id"]
    if eid in processed_ids:
        return                                  # duplicate: swallow silently
    sent.append(eid)
    processed_ids.add(eid)

c = Consumer(b, "mailer3", [0])
for msg in c.poll():
    handle(msg)                                 # process all 5, CRASH before commit
c_new = Consumer(b, "mailer3", [0])
for msg in c_new.poll():
    handle(msg)                                 # full redelivery... all deduped
c_new.commit()
assert sorted(sent) == [f"e{i}" for i in range(5)], \
    "effectively once: at-least-once delivery + idempotence = each effect exactly once"
print(f"effectively-once: {len(sent)} sends despite full redelivery")

This pattern — **retries for liveness, idempotency keys for safety** — is one of the most
transferable ideas in the course: payment APIs, RPC retries (nb 08's "did it happen?"),
stream processors, and capstone 25 all stand on it.

## 3. Time — the hard part

Every event has (at least) two timestamps: **event time** (when it happened, stamped at the
source) and **processing time** (when your system saw it). They diverge — network hiccups,
offline mobile clients, a consumer catching up on backlog — and events arrive **out of
order** in event time. Any computation over time buckets (**windows**) must pick which time
it believes.

Failure first: count clicks in 10-second **tumbling windows** (fixed, non-overlapping
buckets). Users click steadily, but a 20-second network outage delays the delivery of
everything that happened during it:

In [ ]:
events = []
for i in range(3_000):
    t_event = float(rng.uniform(0, 60))                       # when the click happened
    delay = float(rng.uniform(0.1, 2.0))                      # normal delivery
    if 30 <= t_event < 40:                                    # outage: held up 15-25s
        delay = float(rng.uniform(15.0, 25.0))
    events.append({"event_time": t_event, "arrival_time": t_event + delay})
events.sort(key=lambda e: e["arrival_time"])                  # the stream, as observed

WINDOW = 10.0
def window_of(t): return int(t // WINDOW)

truth = defaultdict(int)                       # what actually happened per window
for e in events:
    truth[window_of(e["event_time"])] += 1

# Naive: bucket by PROCESSING time (when it arrived):
by_processing = defaultdict(int)
for e in events:
    by_processing[window_of(e["arrival_time"])] += 1

outage_w = 3                                   # the [30,40) window
print(f"window [30,40): truth={truth[outage_w]}, "
      f"processing-time count={by_processing[outage_w]}")
assert by_processing[outage_w] < truth[outage_w] * 0.5, \
    "processing-time windows: the outage window lost most of its events to later buckets"

The events weren't lost — they were *counted in the wrong windows*, and every dashboard,
bill, and alert built on those counts is wrong. Fixing it means bucketing by **event time**,
which raises the real question: *when is a window finished?* You can't wait forever. The
standard answer is a **watermark** — a heuristic clock that trails the maximum event time
seen by an `allowed_lateness` slack: when the watermark passes a window's end, the window
closes and emits; events for already-closed windows are **late** and go to a side output:

In [ ]:
def event_time_windows(stream, allowed_lateness: float):
    """Tumbling event-time windows with a watermark. Returns (emitted, late)."""
    open_windows: dict[int, int] = defaultdict(int)
    emitted: dict[int, int] = {}
    late: list[dict] = []
    max_event_time = float("-inf")

    for e in stream:                                   # in ARRIVAL order
        max_event_time = max(max_event_time, e["event_time"])
        watermark = max_event_time - allowed_lateness  # 'probably no older events left'
        w = window_of(e["event_time"])
        if w in emitted:
            late.append(e)                             # its window already shipped
        else:
            open_windows[w] += 1
        for w_open in [w for w in open_windows if (w + 1) * WINDOW <= watermark]:
            emitted[w_open] = open_windows.pop(w_open) # watermark passed: close it
    emitted.update(open_windows)                       # end of stream: flush the rest
    return emitted, late

# Generous lateness (30s > the outage delay): perfect event-time counts, nothing late.
emitted, late = event_time_windows(events, allowed_lateness=30.0)
assert emitted == dict(truth) and late == [], \
    "with enough patience, event-time windows reconstruct the truth exactly"

# Tight lateness (3s): results ship fast, but outage stragglers miss their window.
emitted_fast, late_fast = event_time_windows(events, allowed_lateness=3.0)
assert len(late_fast) > 0 and emitted_fast[outage_w] < truth[outage_w]
assert sum(emitted_fast.values()) + len(late_fast) == len(events), \
    "nothing silently disappears: every event is either windowed or flagged late"
print(f"lateness=30s: exact.  lateness=3s: {len(late_fast)} late events "
      f"({100*len(late_fast)/len(events):.1f}%) need a correction path")

That is the entire streaming-time trade in two runs: **watermark slack buys correctness
with latency.** Tight watermarks emit fast and must handle corrections (re-emit updated
windows, or route late data to a reconciliation job — nb 12's lambda architecture fought
exactly this); loose watermarks are accurate and slow. There is no third option, only an
explicit choice.

## 4. Joins and the changelog

**Stream enrichment** (stream-table join): each event needs context — the click needs the
user's current tier. Querying a database per event is nb 01's N+1 again; instead, the
processor *subscribes to the table's changes* (its **changelog**) and maintains a local
copy, joining in memory:

In [ ]:
# One timeline, two kinds of records: user-table changes and click events.
timeline = [
    (1.0, "changelog", {"user": "u1", "tier": "free"}),
    (2.0, "changelog", {"user": "u2", "tier": "free"}),
    (5.0, "click", {"user": "u1", "page": "/pricing"}),
    (8.0, "changelog", {"user": "u1", "tier": "premium"}),      # u1 upgrades
    (10.0, "click", {"user": "u1", "page": "/docs"}),
    (11.0, "click", {"user": "u2", "page": "/docs"}),
]

table: dict[str, str] = {}                     # local materialized user table
enriched = []
for t, kind, rec in timeline:
    if kind == "changelog":
        table[rec["user"]] = rec["tier"]       # upsert: newest change wins
    else:
        enriched.append((t, rec["user"], rec["page"], table.get(rec["user"])))

print(enriched)
assert enriched[0][3] == "free" and enriched[1][3] == "premium", \
    "the join sees the table AS OF each event - u1's clicks straddle the upgrade"

Note the subtlety the assert encodes: this join is **time-dependent**. Reprocessing the
clicks next month against the *final* table would label the 5.0s click "premium" — a
different (wrong) answer. Correct reprocessing must replay table changelog and event stream
on one timeline — which our single `timeline` did by construction. Remember this; it
returns with money on the line as *point-in-time correctness* for ML features (nb 16, 25).

The changelog idea generalizes into two load-bearing patterns:

- **Event sourcing**: make the log the *source of truth* — store immutable facts ("cart
  item added", "item removed"), derive current state by folding over them. State becomes a
  cache of the log; bugs are fixed by re-folding history with better code; and you keep
  information updates destroy (nb 02's "facts about then").
- **Change data capture (CDC)**: databases already have such a log — the replication log
  from nb 05! CDC taps it and publishes every committed change to a broker, turning any
  database into a stream producer. Search indexes, caches, warehouses then follow the
  *same ordered changelog* instead of racing dual writes.

In [ ]:
# Event sourcing in five lines: state = fold(log).
cart_log = [("add", "book"), ("add", "pen"), ("add", "book"),
            ("remove", "pen"), ("add", "lamp")]

def fold(log):
    cart: dict[str, int] = defaultdict(int)
    for op, item in log:
        cart[item] += 1 if op == "add" else -1
    return {k: n for k, n in cart.items() if n > 0}

assert fold(cart_log) == {"book": 2, "lamp": 1}
assert fold(cart_log[:3]) == {"book": 2, "pen": 1}, "any past state is reconstructible"
print(f"state = fold(events): {fold(cart_log)} - the table is a VIEW of the log")

## 5. Exercises

1. **Key design.** A payments topic must guarantee that each account's transactions are
   processed in order, across 64 partitions and 64 consumers. What is the partition key?
   What ordering does a transfer *between* two accounts get, and what has to absorb that?
2. **Lag as a queue.** Define consumer lag from this notebook's primitives (offsets). Your
   lag graph shows steady growth of 2,000 messages/minute. Using nb 01's Little's law and
   the hockey stick, what are the two possible diagnoses and the first metric you'd check
   to distinguish them?
3. **Idempotency keys.** For the email consumer, compare storing `processed_ids` (a) in
   consumer memory, (b) in Redis with a TTL, (c) in the same database transaction as the
   business write. Which crash windows does each leave open?
4. **Window shopping.** Choose tumbling / hopping (sliding) / session windows for: (i) "p99
   latency over the last 5 min, updated every 10 s", (ii) hourly billing counts,
   (iii) "visits" on a website. One sentence each on why.
5. **The dual-write bug.** A service writes an order to Postgres AND publishes
   `order_created` to the broker, in that order, no shared transaction. Enumerate the two
   inconsistent outcomes crashes can produce, then explain how the **outbox pattern**
   (write the event into the DB in the same transaction, let CDC publish it) removes both.

### Solutions

**1.** Key = `account_id` — each account's history lives in one partition, strictly
ordered. A transfer touches *two* accounts whose events land in two partitions with **no
mutual order**; the consumer layer must tolerate seeing the credit before the debit
(reorder buffers, or model the transfer as its own saga/event with both legs inside one
keyed event). Cross-entity ordering is exactly what a log does *not* give you — nb 09's
total order costs consensus.

**2.** Lag(partition) = `len(log) − committed_offset`; total lag = sum. Growth means
arrival rate > processing rate (λ > μ). Diagnoses: (a) traffic rose past capacity — check
produce rate; (b) processing slowed (a slow dependency, GC, a poison-pill message
retry-looping) — check per-message processing time / error rate. The hockey stick warns
you: at ρ→1 lag explodes *before* CPU looks saturated.

**3.** (a) Memory: dies with the process — the crash-before-commit window returns
untouched; only helps within one run. (b) Redis+TTL: survives restarts; leaves (i) the
TTL horizon (older redeliveries duplicate) and (ii) a two-system race — mark-then-crash
loses the email, send-then-crash duplicates the mark... unless you order it
send→mark, accepting rare duplicates. (c) Same transaction as the business effect: the
mark and the effect commit atomically (nb 07) — no window at all; this is the gold
standard and why "exactly-once" sinks are really "transactional dedup at the sink".

**4.** (i) Hopping window (5 min length, 10 s hop) — smooth rolling view. (ii) Tumbling
(1 h, aligned) — every event billed exactly once. (iii) Session windows (gap timeout,
e.g. 30 min) — activity defines the boundary, not the clock (nb 10 exercise 1's batch
version, streamified).

**5.** Crash after DB commit, before publish → order exists, downstream never hears
(silent divergence). Crash after publish... if you publish first: event exists, order
doesn't (phantom order). Any retry logic just picks which side duplicates. Outbox: the
order row AND an `outbox` event row commit in **one** DB transaction (atomicity, nb 07);
a CDC tailer reads the committed outbox and publishes at-least-once; consumer idempotence
(§2) absorbs its duplicates. One source of truth, one ordered changelog — which is the
thesis of the next notebook.

## 6. Takeaways

- The **log** unifies everything: storage engine (nb 03), replication stream (nb 05),
  broker partition (today). Offsets make consumers stateless-restartable; retention makes
  history replayable; partitioning by key buys per-key order and nothing more.
- Delivery is a two-step dance (process, commit) and crashes choose your semantics:
  commit-first loses, process-first duplicates. **At-least-once + idempotence =
  effectively once** — retries for liveness, identity for safety.
- Event time ≠ processing time. Windows need watermarks; watermark slack trades latency
  against completeness, and late data needs a designed path, not denial.
- Tables and streams are two views of one thing: **table = fold(changelog)**. Enrichment
  joins are time-dependent; CDC + outbox turn databases into honest stream producers.

**Further reading.** DDIA ch. 11; Kreps, *The Log: What every software engineer should
know...* (2013) — the essay behind this whole notebook; Akidau et al., *The Dataflow
Model* (VLDB 2015) — watermarks and windows, definitively.

**Next:** [12 · Derived data](12-derived-data.ipynb) — the Part 1 finale: compose
everything into an "unbundled database", and keep an ecosystem of views consistent with
one truth.